# Telco Customer Churn — Data Preprocessing & Feature Engineering







## 1. Import libraries

In [1]:
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_FILE = DATA_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)


Python: c:\Users\Neeraj\Desktop\Data Analytics\Churn_prediction\.venv\Scripts\python.exe
Project root: C:\Users\Neeraj\Desktop\Data Analytics\Churn_prediction


## 2. Load dataset

In [2]:
df = pd.read_csv(RAW_FILE)

print("Shape:", df.shape)
display(df.head())


Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Basic data quality check

In [3]:
print("Duplicate rows:", df.duplicated().sum())
print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).head(10))

print("\nData types:")
display(df.dtypes)


Duplicate rows: 0

Missing values:


customerID         0
gender             0
SeniorCitizen      0
Partner            0
Dependents         0
tenure             0
PhoneService       0
MultipleLines      0
InternetService    0
OnlineSecurity     0
dtype: int64


Data types:


customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

## 4. Clean `TotalCharges`

In [4]:
# TotalCharges contains blank strings in the original dataset.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Customers with tenure = 0 have no accumulated charges.
df.loc[
    df["TotalCharges"].isna() & (df["tenure"] == 0),
    "TotalCharges"
] = 0

# Defensive fallback for any remaining missing values.
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

print("Missing TotalCharges after cleaning:", df["TotalCharges"].isna().sum())
display(df[["tenure", "MonthlyCharges", "TotalCharges"]].head())


Missing TotalCharges after cleaning: 0


,tenure,MonthlyCharges,TotalCharges
0,1,29.85,29.85
1,34,56.95,1889.50
2,2,53.85,108.15
3,45,42.30,1840.75
4,2,70.70,151.65


## 5. Drop `customerID` and encode target

In [5]:
# customerID is an identifier, not a predictive business feature.
df = df.drop(columns=["customerID"])

# Churn: No = 0, Yes = 1
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

print("Columns after dropping customerID:")
print(df.columns.tolist())


Columns after dropping customerID:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## 6. Feature engineering

In [ ]:
# Feature 1: TenureGroup
tenure_bins = [-1, 6, 12, 24, 48, np.inf]
tenure_labels = [
    "0-6 months",
    "7-12 months",
    "13-24 months",
    "25-48 months",
    "49+ months"
]

df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=tenure_bins,
    labels=tenure_labels
)

# Feature 2: Number of subscribed services
service_cols = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["NumServices"] = sum(
    (df[col] == "Yes").astype(int) for col in service_cols
)

# Feature 3: average monthly charge based on total charges and tenure
df["AvgMonthlyCharge"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
).round(2)

display(
    df[[
        "tenure",
        "TenureGroup",
        "NumServices",
        "MonthlyCharges",
        "TotalCharges",
        "AvgMonthlyCharge",
        "Churn"
    ]].head(10)
)


,tenure,TenureGroup,NumServices,MonthlyCharges,TotalCharges,AvgMonthlyCharge,Churn
0,1,0-6 months,1,29.85,29.85,29.85,0
1,34,25-48 months,3,56.95,1889.50,55.57,0
2,2,0-6 months,3,53.85,108.15,54.08,1
3,45,25-48 months,3,42.30,1840.75,40.91,0
4,2,0-6 months,1,70.70,151.65,75.82,1
5,8,7-12 months,5,99.65,820.50,102.56,1
6,22,13-24 months,4,89.10,1949.40,88.61,0
7,10,7-12 months,1,29.75,301.90,30.19,0
8,28,25-48 months,6,104.80,3046.05,108.79,1
9,62,49+ months,3,56.15,3487.95,56.26,0


### Why these features matter

- **TenureGroup:** captures customer lifecycle stage; early-tenure customers can behave differently from long-tenure customers.
- **NumServices:** summarizes service adoption into one interpretable feature.
- **AvgMonthlyCharge:** approximates the customer's historical average monthly spend.


In [7]:
# Save the complete cleaned + engineered dataset
df.to_csv(DATA_DIR / "engineered_features.csv", index=False)

print("Saved:", DATA_DIR / "engineered_features.csv")
print("Engineered shape:", df.shape)


Saved: C:\Users\Neeraj\Desktop\Data Analytics\Churn_prediction\data\engineered_features.csv
Engineered shape: (7043, 23)


## 7. Separate X and y

In [8]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
display(y.value_counts())
display(y.value_counts(normalize=True).rename("proportion"))


X shape: (7043, 22)
y shape: (7043,)

Target distribution:


Churn
0    5174
1    1869
Name: count, dtype: int64

Churn
0    0.73463
1    0.26537
Name: proportion, dtype: float64

## 8. Stratified 80/20 train-test split

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nChurn rate comparison:")
print(f"Overall: {y.mean():.2%}")
print(f"Train  : {y_train.mean():.2%}")
print(f"Test   : {y_test.mean():.2%}")


X_train: (5634, 22)
X_test : (1409, 22)
y_train: (5634,)
y_test : (1409,)

Churn rate comparison:
Overall: 26.54%
Train  : 26.54%
Test   : 26.54%


### Why `stratify=y`?

The churn target is imbalanced. Stratification keeps approximately the same churn/non-churn proportion in both train and test sets, making evaluation more representative.


## 9. Separate numerical and categorical columns

In [10]:
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical columns:")
print(numeric_features)

print("\nCategorical columns:")
print(categorical_features)

print("\nCounts:")
print("Numerical:", len(numeric_features))
print("Categorical:", len(categorical_features))


Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'NumServices', 'AvgMonthlyCharge']

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup']

Counts:
Numerical: 6
Categorical: 16


C:\Users\Neeraj\AppData\Local\Temp\ipykernel_3744\899858914.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


## 10. Build the ColumnTransformer

In [11]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print(preprocessor)


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges', 'NumServices',
                                  'AvgMonthlyCharge']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'Partner', 'Dependents',
                                 

### What the transformer does

**Numerical:** missing values → median → standardization

**Categorical:** missing values → most frequent category → one-hot encoding

`handle_unknown="ignore"` prevents errors if a category appears in future/test data that was not present during training.


## 11. Fit only on training data

In [12]:
# Fit on X_train only to prevent data leakage.
X_train_processed = preprocessor.fit_transform(X_train)

# Only transform X_test using the already-fitted transformer.
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

print("Processed X_train:", X_train_processed_df.shape)
print("Processed X_test :", X_test_processed_df.shape)

display(X_train_processed_df.head())


Processed X_train: (5634, 52)
Processed X_test : (1409, 52)


,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__NumServices,num__AvgMonthlyCharge,cat__gender_Female,cat__gender_Male,cat__Partner_No,cat__Partner_Yes,cat__Dependents_No,cat__Dependents_Yes,cat__PhoneService_No,cat__PhoneService_Yes,cat__MultipleLines_No,cat__MultipleLines_No phone service,cat__MultipleLines_Yes,cat__InternetService_DSL,cat__InternetService_Fiber optic,cat__InternetService_No,cat__OnlineSecurity_No,cat__OnlineSecurity_No internet service,cat__OnlineSecurity_Yes,cat__OnlineBackup_No,cat__OnlineBackup_No internet service,cat__OnlineBackup_Yes,cat__DeviceProtection_No,cat__DeviceProtection_No internet service,cat__DeviceProtection_Yes,cat__TechSupport_No,cat__TechSupport_No internet service,cat__TechSupport_Yes,cat__StreamingTV_No,cat__StreamingTV_No internet service,cat__StreamingTV_Yes,cat__StreamingMovies_No,cat__StreamingMovies_No internet service,cat__StreamingMovies_Yes,cat__Contract_Month-to-month,cat__Contract_One year,cat__Contract_Two year,cat__PaperlessBilling_No,cat__PaperlessBilling_Yes,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__TenureGroup_0-6 months,cat__TenureGroup_13-24 months,cat__TenureGroup_25-48 months,cat__TenureGroup_49+ months,cat__TenureGroup_7-12 months
3738,-0.441773,0.102371,-0.521976,-0.262257,-0.184954,-0.539937,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3151,-0.441773,-0.711743,0.337478,-0.503635,-0.667823,0.391497,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4860,-0.441773,-0.793155,-0.809013,-0.749883,-0.184954,-0.646151,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
3867,-0.441773,-0.263980,0.284384,-0.172722,0.780784,0.276681,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3810,-0.441773,-1.281624,-0.676279,-0.989374,-1.150692,-0.674606,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


## 12. Save clean and model-ready files

In [13]:
# Clean engineered X splits
X_train.to_csv(DATA_DIR / "clean_X_train.csv", index=False)
X_test.to_csv(DATA_DIR / "clean_X_test.csv", index=False)

# Target splits
y_train.to_csv(DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(DATA_DIR / "y_test.csv", index=False)

# Fully transformed model-ready features
X_train_processed_df.to_csv(DATA_DIR / "processed_X_train.csv", index=False)
X_test_processed_df.to_csv(DATA_DIR / "processed_X_test.csv", index=False)

# Save fitted transformer for later prediction/inference.
joblib.dump(preprocessor, PROJECT_ROOT / "src" / "preprocessor.joblib")

print("All preprocessing files saved successfully.")


All preprocessing files saved successfully.


## 13. Final validation

In [14]:
print("Original dataset rows :", len(df))
print("Training rows         :", len(X_train))
print("Testing rows          :", len(X_test))
print("Processed train cols  :", X_train_processed_df.shape[1])
print("Processed test cols   :", X_test_processed_df.shape[1])

assert "customerID" not in X_train.columns
assert len(X_train) + len(X_test) == len(df)
assert abs(y_train.mean() - y_test.mean()) < 0.02

print("\nValidation checks passed.")


Original dataset rows : 7043
Training rows         : 5634
Testing rows          : 1409
Processed train cols  : 52
Processed test cols   : 52

Validation checks passed.
